# SPE Automation — Module Reference

This notebook documents every function needed to run the full automation pipeline. Organised by flowchart step.

**Pipeline order:**
1. Initialisation
2. Coarse scan
3. Classify & visualise
4. Fine scan
5. Long integration spectrum
6. Bandpass filter setup
7. g² acquisition
8. g² analysis

In [1]:
%matplotlib qt
%reload_ext autoreload
%autoreload 2

import sgd
import lf_spec
import pl_spec
import filter
import classifier
import plotter
import picoharp
import g2 as g2mod
from main import run_bandpass_setup, find_emission_fwhm_center, angle_for_wavelength

---
## 1. Initialisation

Run once at the start of every session. All hardware must be powered on before running these cells.

### `lf_spec.lf_connect()`
Starts the LightField bridge process if it isn't already running, then connects to the open LightField instance. Safe to call multiple times — does nothing if already connected.

| Parameter | Type | Description |
|-----------|------|-------------|
| *(none)* | | |

**Raises:** `RuntimeError` if LightField is not open or the bridge fails to start within 60 s.

In [2]:
lf_spec.lf_connect()

LightField bridge already running.


### `sgd.sgd_init()`
Connects to the scanning galvo driver (Siglent SDG) over USB-VISA and zeros both channels. Safe to call multiple times — returns the existing object if already initialised.

| Parameter | Type | Description |
|-----------|------|-------------|
| *(none)* | | |

**Constants:** `XCONV = -1.85/20` V/µm, `YCONV = 2.65/20` V/µm. Max range ±10 V (≈ ±108 µm X, ±75 µm Y).

In [3]:
sgd.sgd_init()

Scanning for instruments...
Getting ready...
Ready for use!


### `filter.filter_init()` · `filter.filter_on()`
Connects to the Thorlabs FilterFlipper (MFF101, serial 37010764) and KCube DC Servo rotation stage (KDC101, serial 27600279) via Kinesis SDK.

Call `filter_init()` then `filter_on()` once per session.

| Function | Parameters | Description |
|----------|-----------|-------------|
| `filter_init()` | *(none)* | Build device list, connect flipper + rotation stage, load motor config |
| `filter_on()` | *(none)* | Enable both devices so they can be commanded |

In [ ]:
filter.filter_init()
filter.filter_on()

### `picoharp.ph_init()`
Loads PHLib64.dll, opens the PicoHarp 300 (device index 0), initialises T2 mode, calibrates, and sets fixed CFD levels (150 mV threshold, 10 mV zero-cross).

| Parameter | Type | Description |
|-----------|------|-------------|
| *(none)* | | |

**Raises:** `RuntimeError` if the device cannot be opened or calibration fails.

In [ ]:
picoharp.ph_init()

---
## 2. Scanning Galvo (SGD)

Controls the mirror position for raster scanning.

### SGD Functions

| Function | Parameters | Description |
|----------|-----------|-------------|
| `sgd_on()` | *(none)* | Enable both output channels |
| `sgd_off()` | *(none)* | Zero offsets and disable channels |
| `set_position(x_um, y_um, silent=False)` | See below | Move mirror to position in µm |
| `goto(x, y)` | `x, y` float µm | `sgd_on()` then `set_position()` |
| `home()` | *(none)* | Move to (0, 0) then `sgd_off()` |

#### `set_position` parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `x_um` | float | X position in micrometres |
| `y_um` | float | Y position in micrometres |
| `silent` | bool | Suppress move/done print messages. Default `False` |

In [ ]:
# Move to a specific position
sgd.goto(-4.5, -11.0)

# Return to centre and power off
sgd.home()

---
## 3. Spectrometer — LightField Direct Interface (`lf_spec`)

All spectrometer control goes through the persistent LightField bridge. Call `lf_connect()` once per session before using any of these.

### `lf_spec.lf_setup()`
Set exposure, center wavelength, and grating in one call.

| Parameter | Type | Description |
|-----------|------|-------------|
| `exposure_s` | float | Exposure time in **seconds**. Default `1` |
| `center_wavelength` | float | Spectrometer centre wavelength in nm. Default `700` |
| `grating` | int or str | `150` → 150 g/mm `[800nm,150][2][0]`; `600` → 600 g/mm `[500nm,600][1][0]`. Default `150` |

**For long scans:** set `center_wavelength = int((532 + emission_wl) / 2)` so both the laser reference (532 nm) and the emission peak fit within the ~130 nm window of the 600 g/mm grating.

### `lf_spec.lf_acquire()`
Trigger a single acquisition. Blocks until the frame arrives (60 s timeout).

| Parameter | Type | Description |
|-----------|------|-------------|
| *(none)* | | |

**Returns:** `(intensity, wavelength)` — both `np.ndarray` of shape `(N,)` where N = number of CCD pixels (typically 1340).

### Other `lf_spec` functions

| Function | Parameters | Returns | Description |
|----------|-----------|---------|-------------|
| `lf_set_exposure(exposure_s)` | `exposure_s` float (seconds) | — | Set exposure time |
| `lf_set_grating(grating)` | `grating` int (150 or 600) or full string | — | Set grating |
| `lf_set_center_wavelength(wl_nm)` | `wl_nm` float | — | Set centre wavelength |
| `lf_get_wavelengths()` | *(none)* | `np.ndarray` | Current wavelength calibration array |
| `lf_reconnect()` | *(none)* | — | Reconnect bridge to LightField after closing/reopening it |
| `lf_shutdown()` | *(none)* | — | Gracefully stop the bridge. **Always call before closing LightField** |

In [ ]:
# Single acquisition example
lf_spec.lf_setup(exposure_s=1, center_wavelength=700, grating=150)
intensity, wl = lf_spec.lf_acquire()
print(f'{len(intensity)} points, {wl[0]:.1f}–{wl[-1]:.1f} nm, peak={intensity.max():.0f}')

---
## 4. PL Raster Scan (`pl_spec`)

Drives the SGD mirror through a grid, acquires a spectrum at each point, and saves results to disk. Runs the classifier automatically on completion.

### `pl_spec.pl_spec_manual()`

| Parameter | Type | Description |
|-----------|------|-------------|
| `xdim` | float | Total scan width in µm. Set `0` for single-point scan |
| `ydim` | float | Total scan height in µm. Set `0` for single-point scan |
| `dx` | float | Step size in X (µm) |
| `dy` | float | Step size in Y (µm) |
| `foldername` | str | Folder name under `data/` where results are saved |
| `current_user` | str | User for out-of-focus Telegram alerts. Options: `'shuhul'`, `'kristina'`, `'holland'` |
| `center` | tuple(float, float) | `(x, y)` centre of scan in µm. Default `(0, 0)` |
| `grating` | int | `150` or `600` g/mm. Default `150` |
| `exposure_time` | float | Exposure per point in **seconds**. Default `1` |
| `center_wavelength` | int | Spectrometer centre wavelength in nm. Default `700` |
| `classification_threshold` | float | Emission peak must exceed `threshold × laser_peak` to be classified. Default `1.05` |
| `scan_type` | str | Subfolder name: `'coarse'`, `'fine'`, `'long_x{x}_y{y}'`. Default `'coarse'` |
| `data_folder` | str | Root data directory. Default `'data'` |
| `eng` | matlab.engine or None | Pass an already-connected MATLAB engine to avoid double-connect. Default `None` |

**Saved files** (in `data/<foldername>/<scan_type>/`):
- `out.npy` — shape `(ny, nx, nwl)` intensity cube
- `wl.npy` — wavelength array
- `xs.npy`, `ys.npy` — scan grid coordinates
- `classified.npy` — binary emitter map (after classifier runs)

In [ ]:
# Coarse scan
foldername = '20260519-PLSPC-example'

pl_spec.pl_spec_manual(
    xdim=1, ydim=1, dx=0.5, dy=0.5,
    center=(0, 0),
    grating=150,
    exposure_time=1,
    center_wavelength=700,
    foldername=foldername,
    scan_type='coarse',
    current_user='shuhul',
)

In [ ]:
# Fine scan centred on a classified emitter
target_x, target_y = -4.5, -11.0

pl_spec.pl_spec_manual(
    xdim=2, ydim=2, dx=0.2, dy=0.2,
    center=(target_x, target_y),
    grating=150,
    exposure_time=3,
    center_wavelength=700,
    foldername=foldername,
    scan_type=f'fine_x{target_x:.1f}_y{target_y:.1f}',
    current_user='shuhul',
)

In [ ]:
# Long integration scan (single point, 600 g/mm, centre_wavelength = midpoint of laser + emission)
emission_wl = 580   # estimated from coarse scan
cwl = int((532 + emission_wl) / 2)

pl_spec.pl_spec_manual(
    xdim=0, ydim=0, dx=0, dy=0,
    center=(target_x, target_y),
    grating=600,
    exposure_time=10,
    center_wavelength=cwl,
    foldername=foldername,
    scan_type=f'long_x{target_x:.1f}_y{target_y:.1f}',
    current_user='shuhul',
)

---
## 5. Classifier (`classifier`)

Identifies single-photon emitters in a scan by finding spectra with a narrow emission peak above the laser line.

### `classifier.classify_all()`
Runs on every pixel in a saved scan and saves `classified.npy`. Called automatically by `pl_spec_manual`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `foldername` | str | Scan folder name under `data/` |
| `scan_type` | str | Subfolder name (e.g. `'coarse'`) |
| `data_folder` | str | Root data directory. Default `'data'` |
| `threshold` | float | Emission/laser peak ratio threshold. Default `1.05` |
| `spatial_radius_um` | float | Distance below which two detections are considered the same emitter. Default `0.8` µm |
| `wl_diff_threshold` | float | Wavelength difference below which two peaks are the same emitter. Default `6.0` nm |

### `classifier.classify_spectrum()`
Classify a single spectrum. Returns `(label, peak_height, peak_wl)`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `spectrum` | np.ndarray | 1-D intensity array |
| `wl` | np.ndarray | Wavelength array |
| `ratio_threshold` | float | Emission/laser threshold. Default `1.05` |

**Returns:** `(int, float or None, float or None)` — `(1=emitter/0=not, peak_height, peak_wavelength_nm)`

In [ ]:
import numpy as np

# Re-run classifier with custom threshold
classifier.classify_all(foldername, 'coarse', threshold=0.8)

# Check results
classified = np.load(f'data/{foldername}/coarse/classified.npy')
xs = np.load(f'data/{foldername}/coarse/xs.npy')
ys = np.load(f'data/{foldername}/coarse/ys.npy')
iys, ixs = np.where(classified == 1)
for iy, ix in zip(iys, ixs):
    print(f'Emitter at x={xs[ix]:.2f}, y={ys[iy]:.2f}')

---
## 6. Plotter (`plotter`)

### `plotter.plot_heatmap_manual()`
Interactive heatmap + spectrum viewer. Heatmap colour = peak intensity above 550 nm.

| Parameter | Type | Description |
|-----------|------|-------------|
| `foldername` | str | Scan folder name |
| `scan_type` | str | Subfolder name |
| `title` | str | Plot title. Default `'PL Spectrum'` |
| `xlabel` | str | X axis label. Default `'X Position (um)'` |
| `ylabel` | str | Y axis label. Default `'Y Position (um)'` |
| `heatmap_cmap` | str | Matplotlib colormap. Default `'viridis'` |
| `class1_color` | str | Colour for classified emitter markers. Default `'red'` |
| `data_folder` | str | Root data directory. Default `'data'` |
| `slideshow_interval` | float | Slideshow step interval in seconds. Default `0.5` |

**Controls:** hover to show spectrum · click to lock · space to play/pause slideshow · arrow keys to step

In [ ]:
plotter.plot_heatmap_manual(foldername, 'coarse')

---
## 7. Bandpass Filter (`filter`)

Controls the Thorlabs flipper mount and rotation stage. **Always shut down with `filter_off()` at end of session.**

### Filter functions

| Function | Parameters | Description |
|----------|-----------|-------------|
| `filter_init()` | *(none)* | Connect to flipper + rotation stage |
| `filter_on()` | *(none)* | Enable both devices |
| `filter_off()` | *(none)* | Disable and disconnect both devices |
| `flip_up()` | *(none)* | **Insert** filter into beam |
| `flip_down()` | *(none)* | **Remove** filter from beam |
| `rotation_home()` | *(none)* | Home the rotation stage. Direction chosen automatically from last saved position to avoid wire wrap |
| `rotation_move(target)` | `target` float (degrees) | Rotate to target angle. Saves position to `rotation_last_pos.txt` |
| `set_rotation_mode(mode)` | `mode` str: `'quickest'`/`'forwards'`/`'reverse'` | Set direction behaviour for `MoveTo` calls. Default after homing: `'quickest'` |

### Bandpass alignment — `run_bandpass_setup()` (from `main`)

Full alignment pipeline: inserts filter, rotates to calibration angle for the target wavelength, acquires a spectrum, checks FWHM centre is within tolerance. Retries with proportional angle correction up to `max_attempts` times.

| Parameter | Type | Description |
|-----------|------|-------------|
| `target_wl` | float | Target wavelength in nm (FWHM centre from long scan) |
| `cal_folder` | str | Calibration subfolder under `calibration/` |
| `current_user` | str | Telegram user for failure notification |
| `tolerance_nm` | float | Acceptable alignment error in nm. Default `2.0` |
| `max_attempts` | int | Retry attempts before giving up. Default `3` |

**Returns:** `True` if aligned within tolerance, `False` if failed (sends Telegram notification and removes filter).

### Helper functions (from `main`)

| Function | Parameters | Returns | Description |
|----------|-----------|---------|-------------|
| `find_emission_fwhm_center(spectrum, wl, laser_cutoff_nm=560)` | spectrum, wl arrays | float (nm) or None | FWHM centre of brightest peak above `laser_cutoff_nm` |
| `angle_for_wavelength(cal_folder, target_wl)` | str, float | float (degrees) or None | Nearest calibration angle for `target_wl` |

In [ ]:
import numpy as np
from main import find_emission_fwhm_center, run_bandpass_setup

scan_type = f'long_x{target_x:.1f}_y{target_y:.1f}'
out = np.load(f'data/{foldername}/{scan_type}/out.npy')
wl  = np.load(f'data/{foldername}/{scan_type}/wl.npy')

target_wl = find_emission_fwhm_center(out[0, 0, :], wl)
print(f'ZPL FWHM centre: {target_wl:.2f} nm')

cal_folder = '2026-04-07_14-48-20'   # update to your latest calibration run
aligned = run_bandpass_setup(target_wl, cal_folder, current_user='shuhul')
print('Aligned:', aligned)

---
## 8. g² Acquisition (`picoharp`)

### `picoharp.ph_acquire()`
Run T2 TTTR acquisition until `target_records` collected or `acq_time_s` elapsed. Parses raw records and saves photon arrival times to a timestamped `.npz`.

| Parameter | Type | Description |
|-----------|------|-------------|
| `target_records` | int | Stop when this many TTTR records collected |
| `acq_time_s` | int | Hard time limit in seconds. Default `10000` |
| `out_folder` | str | Directory to save output `.npz`. Default `'g2_data'` |
| `progress_signal` | PyQt signal or None | Optional GUI progress signal (emits 0–99). Default `None` |

**Returns:** path to saved `.npz` file (contains arrays `ch0`, `ch1` — absolute photon times in ps), or `None` if no data.

### `picoharp.get_count_rates()`
Returns `(ch0_cps, ch1_cps)` — live count rates in counts per second. Useful for checking alignment before starting acquisition.

### `picoharp.ph_close()`
Disconnect from the PicoHarp. Call at end of session.

In [ ]:
# Check count rates before acquiring
r0, r1 = picoharp.get_count_rates()
print(f'Ch0: {r0:.2e} cps   Ch1: {r1:.2e} cps')

# Acquire
npz_path = picoharp.ph_acquire(target_records=500_000, out_folder='g2_data')
print(f'Saved: {npz_path}')

---
## 9. g² Analysis (`g2`)

### `g2.run()` — full pipeline
Parse file → compute g²(τ) → save `.npz` + `.png`. Accepts both `.ptu` and `.npz` files.

| Parameter | Type | Description |
|-----------|------|-------------|
| `path` | str | Path to `.npz` (from `ph_acquire`) or `.ptu` file |
| `out_folder` | str | Output directory. Default `'g2_data'` |
| `g2time_ns` | float | Correlation half-window in ns. Default `100.0` |
| `timebin_ns` | float | Histogram bin width in ns. Default `1.0` |
| `seed` | int | Random seed for afterflash removal. Default `0` |

**Returns:** result dict with keys:

| Key | Description |
|-----|-------------|
| `tau` | Delay axis array (ns) |
| `g2` | Normalised g²(τ) array |
| `popt` | Fit parameters `(a, b, T1, T2)` or `None`. **g²(0) = 1 − b** · **T1 = lifetime estimate (ns)** |
| `N1`, `N2` | Photon counts per channel |
| `cavg` | Background coincidence level |

### `g2.eff2_from_npz()` — compute only (no save)

| Parameter | Type | Description |
|-----------|------|-------------|
| `npz_path` | str | Path to `.npz` from `ph_acquire` |
| `g2time_ns` | float | Correlation half-window in ns. Default `100.0` |
| `timebin_ns` | float | Bin width in ns. Default `1.0` |
| `seed` | int | Random seed. Default `0` |

In [ ]:
# Run full g2 analysis
result = g2mod.run(npz_path, out_folder='g2_data')

if result['popt'] is not None:
    a, b, T1, T2 = result['popt']
    print(f"g²(0) = {1 - b:.3f}")
    print(f"Lifetime T1 = {T1:.1f} ns")
else:
    print('Fit did not converge.')

---
## 10. Shutdown

Run at the end of every session. **Always shut down the bridge before closing LightField.**

In [ ]:
lf_spec.lf_shutdown()   # bridge calls Dispose() — must come before closing LightField
filter.filter_off()
picoharp.ph_close()
sgd.home()